## Purpose of the notebook
Trying out saving data to duckdb database after getting the mapping from aliases to canonical names

In [1]:
import duckdb
import pandas as pd
import logging
from lxml import etree
import gzip

In [2]:
compressed_path = "../data/raw/dblp.xml.gz"
log_file_path = "../logs/duckdb.log"
db_path = "../data/db/dblp.duckdb"


In [3]:
ROOT_TAGS = {
    'article', 'inproceedings', 'proceedings', 'book', 
    'incollection', 'phdthesis', 'mastersthesis', 'www'
}

In [4]:
logging.basicConfig(level=logging.INFO, filename=log_file_path, filemode="w",
                    format="%(asctime)s - %(levelname)s - %(message)s")

In [5]:
WORKS = 200
PEOPLE = 200

In [6]:
def safe_text(elem, tag_name):
    value = elem.findtext(tag_name)
    if value is None:
        return None
    value = value.strip()
    return value or None


def canonicalize_name(name, alias_map):
    name = name.strip()
    return alias_map.get[name]

In [7]:
def canonicalize_names_in_duckdb(conn):
    """
    Build person rows from the small sample of names collected in sample_names.
    No large Python alias dictionary needed.
    """
    conn.execute("DROP TABLE IF EXISTS person")

    conn.execute("""
        CREATE TABLE person (
            person_id INTEGER PRIMARY KEY,
            canonical_name TEXT UNIQUE
        );
    """)

    conn.execute("""
        INSERT INTO person
        SELECT
            row_number() OVER (ORDER BY canonical_name) AS person_id,
            canonical_name
        FROM (
            SELECT DISTINCT
                COALESCE(aa.canonical_name, sn.name) AS canonical_name
            FROM sample_names sn
            LEFT JOIN author_aliases aa
                ON aa.alias_name = sn.name
        ) x
        ORDER BY canonical_name;
    """)

Create only the works and person tables since they're independant

In [8]:
def build_sample_person_and_work_tables(max_works=200, db_path=db_path, compressed_path=compressed_path):
    conn = duckdb.connect(db_path)

    conn.execute("DROP TABLE IF EXISTS work")
    conn.execute("DROP TABLE IF EXISTS person")
    conn.execute("DROP TABLE IF EXISTS sample_names")

    conn.execute("""
        CREATE TEMP TABLE sample_names (
            name TEXT
        );
    """)

    conn.execute("""
        CREATE TABLE work (
            work_id INTEGER PRIMARY KEY,
            dblp_key TEXT UNIQUE,
            record_type TEXT,
            title TEXT,
            year INTEGER,
            mdate TEXT,
            volume TEXT,
            number TEXT,
            pages TEXT,
            journal TEXT,
            booktitle TEXT,
            publisher TEXT,
            school TEXT,
            series TEXT,
            crossref TEXT,
            publtype TEXT,
            url TEXT,
            ee TEXT,
            isbn TEXT,
            raw_xml TEXT
        );
    """)

    work_rows = []
    sampled_names = set()

    with gzip.open(compressed_path, "rb") as f:
        context = etree.iterparse(
            f,
            events=("end",),
            tag=ROOT_TAGS,
            load_dtd=True,
            resolve_entities=True,
            no_network=True,
        )

        for _, elem in context:
            try:
                if elem.tag != "www" and len(work_rows) < max_works:
                    raw_xml = etree.tostring(elem, encoding="unicode")

                    work_rows.append((
                        len(work_rows) + 1,
                        elem.get("key"),
                        elem.tag,
                        safe_text(elem, "title"),
                        int(y) if (y := safe_text(elem, "year")) and y.isdigit() else None,
                        elem.get("mdate"),
                        safe_text(elem, "volume"),
                        safe_text(elem, "number"),
                        safe_text(elem, "pages"),
                        safe_text(elem, "journal"),
                        safe_text(elem, "booktitle"),
                        safe_text(elem, "publisher"),
                        safe_text(elem, "school"),
                        safe_text(elem, "series"),
                        safe_text(elem, "crossref"),
                        elem.get("publtype"),
                        safe_text(elem, "url"),
                        safe_text(elem, "ee"),
                        safe_text(elem, "isbn"),
                        raw_xml,
                    ))

                    for role in ("author", "editor"):
                        for node in elem.findall(role):
                            if node.text:
                                sampled_names.add(node.text.strip())

                    if len(work_rows) >= max_works:
                        break

            finally:
                elem.clear()
                while elem.getprevious() is not None:
                    del elem.getparent()[0]

    conn.executemany(
        "INSERT INTO sample_names VALUES (?)",
        [(name,) for name in sorted(sampled_names)]
    )

    conn.executemany(
        """
        INSERT INTO work (
            work_id, dblp_key, record_type, title, year, mdate,
            volume, number, pages, journal, booktitle, publisher,
            school, series, crossref, publtype, url, ee, isbn, raw_xml
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        work_rows
    )

    canonicalize_names_in_duckdb(conn)
    val = conn.execute('SELECT COUNT(*) FROM person').fetchone()
    if val is not None:
        logging.info(f"Inserted {len(work_rows)} sample works")
        logging.info(f"Inserted {val[0]} sample people")

    return conn

In [9]:
conn = build_sample_person_and_work_tables()


raw_xml is a column only for debugging purposes

In [12]:
conn.execute("SELECT * FROM work LIMIT 10").fetchdf()

,work_id,dblp_key,record_type,title,year,mdate,volume,number,pages,journal,booktitle,publisher,school,series,crossref,publtype,url,ee,isbn,raw_xml
0,1,ms/Hoffmann2008,mastersthesis,Regelbasierte Extraktion und asymmetrische Fus...,2009,2020-03-12,None,None,None,None,None,None,University of Trier,None,None,None,None,http://dblp.uni-trier.de/papers/DiplomarbeitOl...,None,"<mastersthesis mdate=""2020-03-12"" key=""ms/Hoff..."
1,2,ms/Ley2006,mastersthesis,Der Einfluss kleiner naturnaher Retentionsmaßn...,2006,2020-03-12,None,None,None,None,None,None,"Diplomarbeit, Universität Trier, FB VI, Physis...",None,None,None,None,http://dblp.uni-trier.de/papers/DiplomarbeitRi...,None,"<mastersthesis mdate=""2020-03-12"" key=""ms/Ley2..."
2,3,ms/Brown92,mastersthesis,PRPL: A Database Workload Specification Langua...,1992,2018-06-13,None,None,None,None,None,None,University of Wisconsin-Madison,None,None,None,None,None,None,"<mastersthesis mdate=""2018-06-13"" key=""ms/Brow..."
3,4,ms/Vollmer2006,mastersthesis,Portierung des DBLP-Systems auf ein relational...,2006,2018-06-13,None,None,None,None,None,None,"Diplomarbeit, Universität Trier, FB IV, Inform...",None,None,None,None,http://dbis.uni-trier.de/Diplomanden/Vollmer/v...,None,"<mastersthesis mdate=""2018-06-13"" key=""ms/Voll..."
4,5,ms/Klaas2007,mastersthesis,Who's Who in the World Wide Web: Approaches to...,2007,2020-03-12,None,None,None,None,None,None,"Diplomarbeit, LMU München, Informatik",None,None,None,None,http://www.pms.ifi.lmu.de/publikationen/diplom...,None,"<mastersthesis mdate=""2020-03-12"" key=""ms/Klaa..."
5,6,ms/Yurek97,mastersthesis,Efficient View Maintenance at Data Warehouses.,1997,2018-06-13,None,None,None,None,None,None,"University of California at Santa Barbara, Dep...",None,None,None,None,None,None,"<mastersthesis mdate=""2018-06-13"" key=""ms/Yure..."
6,7,series/ifip/computerization1985,book,"Computerization and Work, A Reader on Social A...",1985,2017-05-16,None,None,None,None,Computerization and Work,Springer,None,IFIP State-of-the-Art Reports,None,None,db/series/ifip/computerization1985.html,https://doi.org/10.1007/978-3-642-70453-6,978-3-540-15367-2,"<book mdate=""2017-05-16"" key=""series/ifip/comp..."
7,8,series/ifip/ParkerD14,incollection,Computers in Schools in the USA: A Social Hist...,2014,2023-06-26,None,None,203-211,None,Reflections on the History of Computers in Edu...,None,None,None,series/ifip/hedu2014,None,db/series/ifip/hedu2014.html#ParkerD14,https://doi.org/10.1007/978-3-642-55119-2_14,None,"<incollection mdate=""2023-06-26"" key=""series/i..."
8,9,series/ifip/RheingansL95,incollection,Perceptual Principles for Effective Visualizat...,1995,2017-05-16,None,None,59-73,None,Perceptual Issues in Visualization,None,None,None,series/ifip/piv1995,None,db/series/ifip/piv1995.html#RheingansL95,https://doi.org/10.1007/978-3-642-79057-7_6,None,"<incollection mdate=""2017-05-16"" key=""series/i..."
9,10,series/ifip/ButlerV14,incollection,The Hopes and Realities of the Computer as a S...,2014,2018-06-26,None,None,197-202,None,Reflections on the History of Computers in Edu...,None,None,None,series/ifip/hedu2014,None,db/series/ifip/hedu2014.html#ButlerV14,https://doi.org/10.1007/978-3-642-55119-2_13,None,"<incollection mdate=""2018-06-26"" key=""series/i..."


In [30]:
conn.execute("SELECT * FROM author_aliases WHERE canonical_name = 'Benjamin Müller 0001' LIMIT 20").fetchdf()

,alias_name,canonical_name
0,Benjamin Müller 0001,Benjamin Müller 0001
1,Benjamin Mueller 0001,Benjamin Müller 0001


In [31]:
conn.execute("SELECT * FROM person WHERE canonical_name = 'Benjamin Müller 0001' LIMIT 50").fetchdf()

,person_id,canonical_name
0,39,Benjamin Müller 0001
